# Coffee Standard J25 v2 — rebuild dan visual audit
Mengarantina identity dengan anotasi sibling yang tidak konsisten, memilih visual medoid untuk identity lain, lalu membuat ulang contact sheet. **Tidak ada training atau inference model.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, importlib, json, os, shutil, subprocess, sys
from pathlib import Path
REPO=Path('/content/coffee-bean-detection')
BRANCH='codex/coffee-standard-primary-audit'
REMOTE='https://'+'github.com/ediprin/coffee-bean-detection.git'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REMOTE,str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.analysis.coffee_standard_j25_visual_audit import audit_coffee_standard_j25_visuals
from coffee_detector.analysis.public_dataset_eligibility import extract_audit_archive
from coffee_detector.data.prepare_coffee_standard_primary_v2 import prepare_coffee_standard_primary_v2
from coffee_detector.drive_project import resolve_drive_project_root
PROJECT=resolve_drive_project_root(required_relative_paths=('bundles/coffee-detection-with-standard-v8-yolov8.tar',))
ARCHIVE=PROJECT/'bundles/coffee-detection-with-standard-v8-yolov8.tar'
EXPECTED='5529de365ad888406b5534a5d1bf5a4a29c9937bc095b16174f8516d396bb1fc'
actual=hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
if actual!=EXPECTED: raise RuntimeError(f'SHA256 bundle salah: {actual}')
print('REPO:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('BUNDLE VERIFIED:',actual)


In [ ]:
RAW=extract_audit_archive(ARCHIVE,Path('/content/coffee-standard-v8-raw'))
GROUPED=PROJECT/'datasets/coffee-standard-j25-grouped-v2'
result=prepare_coffee_standard_primary_v2(RAW,GROUPED,seed=42,link_mode='auto')
print('STATUS:',result['status'])
print('SOURCE DERIVATIVES:',result['source_derivative_images'])
print('SOURCE IDENTITIES:',result['source_identity_components'])
print('QUARANTINED IDENTITIES:',result['quarantined_identity_components'])
print('QUARANTINED SOURCE IMAGES:',result['quarantined_source_images'])
print('SELECTED:',result['selected_representatives'])
print('SPLITS:',result['images_by_split'])
print('MISSING:',result['missing_classes_by_split'])
print('TECHNICAL GATES:',result['technical_gates'])
print('TECHNICAL READY:',result['technical_split_ready'])
print('TRAINING AUTHORIZED:',result['training_authorized'])
print('SUMMARY:',GROUPED/'coffee_standard_j25_v2_summary.json')
if not result['technical_split_ready']: raise RuntimeError('STOP: J25 v2 tidak lolos technical gate')


In [ ]:
EVIDENCE=PROJECT/'evidence/coffee-standard-j25-visual-audit-v2'
audit=audit_coffee_standard_j25_visuals(GROUPED,RAW,EVIDENCE,samples_per_class=3,flagged_limit=40)
print('DECISION:',audit['decision'])
print('BUILD FORMAT:',audit['grouped_build_format'])
print('SELECTION:',audit['grouped_selection_policy'])
print('QUARANTINED:',audit['quarantined_identity_components'])
print('SELECTED CLASS OBJECTS:',audit['selected_class_review_objects'])
print('GEOMETRY FLAGS:',audit['geometry_flag_objects'])
print('TRAINING AUTHORIZED:',audit['training_authorized'])
print('SUMMARY:',audit['summary'])


In [ ]:
from IPython.display import Image as DisplayImage, display
for split,path in audit['class_review_sheets'].items():
    print('CLASS REVIEW:',split,path)
    display(DisplayImage(filename=path,width=1400))
print('GEOMETRY FLAGS:',audit['geometry_flag_sheet'])
display(DisplayImage(filename=audit['geometry_flag_sheet'],width=1400))
print('Kirim output rebuild, tiga class sheet, dan geometry sheet. Jangan training.')
